<a href="https://colab.research.google.com/github/LamisAbdallah/AI-UniversityTimeTable-Project/blob/main/WebScrapy_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scrapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.7/331.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 8.2 MB/s eta 0:00:00


In [ ]:
!scrapy startproject books

New Scrapy project 'books', using template directory '/usr/local/lib/python3.12/dist-packages/scrapy/templates/project', created in:
    /content/books

You can start your first spider with:
    cd books
    scrapy genspider example example.com


In [ ]:
%cd books

/content/books


In [ ]:
!scrapy genspider book_spider books.toscrape.com

Created spider 'book_spider' using template 'basic' in module:
  books.spiders.book_spider


In [ ]:
!ls books/spiders

book_spider.py	__init__.py  __pycache__


In [ ]:
import scrapy


class BookSpider(scrapy.Spider):
    name = "book_spider"
    allowed_domains = ["books.toscrape.com"]

    start_urls = [
        f"https://books.toscrape.com/catalogue/page-{i}.html"
        for i in range(1, 51)
    ]

    def parse(self, response):

        page_number = response.url.split("-")[-1].replace(".html", "")

        books = response.css("article.product_pod")

        for book in books:

            name = book.css("h3 a::attr(title)").get()
            price = book.css("p.price_color::text").get()

            stock = book.css("p.instock.availability::text").getall()
            stock = "".join(stock).strip()

            relative_url = book.css("h3 a::attr(href)").get()

            # 🔥 أهم سطر (تصحيح الرابط)
            absolute_url = response.urljoin(relative_url)

            yield scrapy.Request(
                absolute_url,
                callback=self.parse_book,
                meta={
                    "page": page_number,
                    "name": name,
                    "price": price,
                    "stock": stock
                }
            )

    def parse_book(self, response):

        description = response.css(
            "meta[name='description']::attr(content)"
        ).get()

        yield {
            "Page Number": response.meta["page"],
            "Book Name": response.meta["name"],
            "Price": response.meta["price"],
            "Stock Availability": response.meta["stock"],
            "Description": description
        }

In [ ]:
!scrapy crawl book_spider -o books_data.csv

2026-03-04 04:18:53 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: books)
2026-03-04 04:18:53 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.0',
 'Twisted': '25.5.0',
 'Python': '3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}
2026-03-04 04:18:53 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-03-04 04:18:53 [asyncio] DEBUG: Using selector: EpollSelector
2026-03-04 04:18:53 [scrapy.addons] INFO: Enabled addons:
[]
2026-03-04 04:18:53 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-03-04 04:18:53 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-03-04 04:18:53 [scrapy.extensions.telnet] INFO: Telnet Password: 92d28c441f87c988
2026-03-04 04:18:53 [scrapy.

In [ ]:
from google.colab import files
files.download("books_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>